# M1.5 · Softmax regression ve log-sum-exp

Çok sınıflı sınıflandırmanın standart kafası. Üç logit'te softmax, cross-entropy ve gradyanı elle hesaplıyor; taşmayan log-sum-exp yazıyor; CS231n'in klasik vektörize softmax kaybını NumPy'da kuruyor; PyTorch'ta `F.cross_entropy`'yi kendin yeniden yazıyor ve en sonda softmax Jacobian'larını ve confusion matrix'i döngüsüz üretiyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m1-5-softmax.html](https://m-gonul.github.io/pytorch-lab/m1-5-softmax.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m1_5_softmax.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    e = [math.exp(2), math.exp(1), 1.0]\n    ref = [v / sum(e) for v in e]\n    R.test(\"softmax([2,1,0])\", lambda: close(need(ns, \"softmax\")([2, 1, 0]), ref))\n    R.test(\"cross_entropy([2,1,0], 0) = 0.4076\", lambda: close(need(ns, \"cross_entropy\")([2, 1, 0], 0), math.log(sum(e)) - 2))\n\n    def stable():\n        close(need(ns, \"softmax\")([1000, 999, 998]), ref, what=\"z + 998\")\n        close(need(ns, \"logsumexp\")([1000, 999, 998]), 998 + math.log(sum(e)), what=\"LSE\")\n        close(need(ns, \"softmax\")([-1000, -1001]), [1 / (1 + math.exp(-1)), 1 / (1 + math.e)], what=\"büyük negatif\")\n        close(need(ns, \"cross_entropy\")([0, 800], 0), 800.0, what=\"CE, doğru sınıf çok gerideyken\")\n    R.test(\"taşma yok (±1000)\", stable)\n\n    def grad():\n        rng = random.Random(0)\n        for _ in range(5):\n            z = [rng.uniform(-3, 3) for _ in range(4)]\n            y = rng.randrange(4)\n            g = need(ns, \"grad_z\")(z, y)\n            h = 1e-6\n            num = []\n            for k in range(4):\n                zp, zm = list(z), list(z)\n                zp[k] += h; zm[k] -= h\n                num.append((need(ns, \"cross_entropy\")(zp, y) - need(ns, \"cross_entropy\")(zm, y)) / (2 * h))\n            close(g, num, rtol=1e-5, atol=1e-7)\n            ensure(abs(sum(g)) < 1e-9, \"gradyan bileşenleri toplamı 0 olmalı\")\n    R.test(\"grad_z = p − e_y, sayısal türevle aynı, toplamı 0\", grad)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"softmax_rows/softmax_loss: döngü yok\", lambda: (max_loops(src, 0, func=\"softmax_rows\"), max_loops(src, 0, func=\"softmax_loss\")))\n    R.test(\"fit_softmax: yalnız adım döngüsü\", lambda: max_loops(src, 1, func=\"fit_softmax\"))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(2)\n\n    def sm():\n        Z = rng.normal(size=(6, 4)) * 5\n        Z[0] += 1000\n        with np.errstate(over=\"raise\", invalid=\"raise\"):\n            P = need(ns, \"softmax_rows\")(Z)\n        E = np.exp(Z - Z.max(1, keepdims=True))\n        close(P, E / E.sum(1, keepdims=True))\n    R.test(\"softmax_rows kararlı (+1000 satırı dahil)\", sm)\n\n    def loss():\n        X, y = rng.normal(size=(20, 5)), rng.integers(0, 3, 20)\n        W = rng.normal(size=(5, 3)) * 0.1\n        L, dW = need(ns, \"softmax_loss\")(W, X, y, reg=0.1)\n        Z = X @ W\n        lse = np.log(np.exp(Z).sum(1))\n        close(L, (lse - Z[np.arange(20), y]).mean() + 0.1 * (W ** 2).sum(), what=\"kayıp\")\n        h, num = 1e-6, np.zeros_like(W)\n        for i in range(5):\n            for j in range(3):\n                E = np.zeros_like(W); E[i, j] = h\n                num[i, j] = (need(ns, \"softmax_loss\")(W + E, X, y, 0.1)[0] - need(ns, \"softmax_loss\")(W - E, X, y, 0.1)[0]) / (2 * h)\n        close(dW, num, rtol=1e-5, atol=1e-7, what=\"dW (sayısal gradyan)\")\n    R.test(\"softmax_loss: kayıp ve dW (reg dahil) sayısal gradyanla aynı\", loss)\n\n    def fit():\n        centers = np.array([[0, 3], [-3, -2], [3, -2]])\n        y = np.repeat(np.arange(3), 60)\n        X = np.column_stack([centers[y] + rng.normal(size=(180, 2)), np.ones(180)])\n        W = need(ns, \"fit_softmax\")(X, y, 3)\n        acc = (np.argmax(X @ W, 1) == y).mean()\n        ensure(acc >= 0.9, f\"doğruluk {acc:.2f}\")\n        return f\"3 sınıf, doğruluk {acc:.3f}\"\n    R.test(\"fit_softmax 3 kümeyi ayırıyor\", fit)\n",
 "d": "import ast\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n\n    def nobuiltin():\n        fn = next((n for n in ast.walk(ast.parse(src)) if isinstance(n, ast.FunctionDef) and n.name == \"ce_from_logits\"), None)\n        ensure(fn is not None, \"`ce_from_logits` bulunamadı\")\n        names = {n.attr for n in ast.walk(fn) if isinstance(n, ast.Attribute)} | {n.id for n in ast.walk(fn) if isinstance(n, ast.Name)}\n        ensure(not ({\"cross_entropy\", \"log_softmax\", \"CrossEntropyLoss\", \"nll_loss\", \"softmax\"} & names), \"ce_from_logits'i hazır fonksiyonlarla yazma\")\n    R.test(\"ce_from_logits hazır CE/softmax kullanmıyor\", nobuiltin)\n    g = torch.Generator().manual_seed(0)\n    Z = torch.randn(16, 5, generator=g, dtype=torch.float64) * 3\n    y = torch.randint(0, 5, (16,), generator=g)\n\n    def val():\n        f = need(ns, \"ce_from_logits\")\n        close(f(Z, y), F.cross_entropy(Z, y))\n        Zb = Z.clone(); Zb[0] += 1e4; Zb[1, 2] = -1e4\n        v = f(Zb, y)\n        ensure(torch.isfinite(v), \"büyük logit'lerde sonuç sonlu değil\")\n        close(v, F.cross_entropy(Zb, y), what=\"±10⁴ logit\")\n    R.test(\"F.cross_entropy ile aynı, ±10⁴'te sonlu\", val)\n\n    def grad():\n        Zg = Z.clone().requires_grad_(True)\n        need(ns, \"ce_from_logits\")(Zg, y).backward()\n        close(Zg.grad, (torch.softmax(Z, 1) - F.one_hot(y, 5)) / 16, what=\"∂L/∂Z\")\n    R.test(\"autograd gradyanı (P − Y)/n\", grad)\n\n    def tr():\n        centers = torch.tensor([[0.0, 3.0], [-3.0, -2.0], [3.0, -2.0], [0.0, -5.0]])\n        yy = torch.arange(4).repeat_interleave(50)\n        X = centers[yy] + torch.randn(200, 2, generator=g)\n        torch.manual_seed(0)\n        m = need(ns, \"train_softmax\")(X, yy, 4)\n        ensure(isinstance(m, nn.Linear) and m.out_features == 4, \"nn.Linear(d, 4) bekleniyordu\")\n        acc = (m(X).argmax(1) == yy).float().mean().item()\n        ensure(acc >= 0.85, f\"doğruluk {acc:.2f}\")\n        return f\"4 sınıf, doğruluk {acc:.3f}\"\n    R.test(\"train_softmax\", tr)\n",
 "e": "import torch\nfrom torch.func import jacrev, vmap\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n    Z = torch.randn(32, 6, generator=g, dtype=torch.float64)\n\n    def jac():\n        J = need(ns, \"softmax_jacobian\")(Z)\n        ensure(tuple(J.shape) == (32, 6, 6), f\"shape {tuple(J.shape)}\")\n        close(J, vmap(jacrev(lambda z: torch.softmax(z, 0)))(Z), what=\"vmap(jacrev(softmax)) ile\")\n        close(J.sum(2), torch.zeros(32, 6, dtype=torch.float64), atol=1e-12, what=\"satır toplamları (Σ p = 1 olduğundan 0)\")\n        return \"J simetrik ve satır toplamları 0\"\n    R.test(\"softmax_jacobian = vmap(jacrev(softmax))\", jac)\n\n    def cm():\n        f = need(ns, \"confusion_matrix\")\n        C = f(torch.tensor([0, 0, 1, 2, 2]), torch.tensor([0, 1, 1, 2, 0]), 3)\n        ensure(C.tolist() == [[1, 1, 0], [0, 1, 0], [1, 0, 1]], f\"{C.tolist()}\")\n        yt = torch.randint(0, 7, (500,), generator=g); yp = torch.randint(0, 7, (500,), generator=g)\n        ref = torch.zeros(7, 7, dtype=torch.long)\n        for a, b in zip(yt.tolist(), yp.tolist()):\n            ref[a, b] += 1\n        ensure(torch.equal(f(yt, yp, 7).long(), ref), \"rastgele 7 sınıfta yanlış\")\n        C2 = f(torch.tensor([0, 0]), torch.tensor([0, 0]), 4)\n        ensure(tuple(C2.shape) == (4, 4), \"hiç görünmeyen sınıflar da matriste olmalı (minlength)\")\n    R.test(\"confusion_matrix: elle örnek, rastgele, eksik sınıflar\", cm)\n    if \"softmax_jacobian\" not in ns or \"confusion_matrix\" not in ns:\n        return\n    Zb = torch.randn(2000, 10, generator=g)\n    yt = torch.randint(0, 10, (100_000,), generator=g); yp = torch.randint(0, 10, (100_000,), generator=g)\n\n    def loopJ():\n        out = []\n        for z in Zb:\n            p = torch.softmax(z, 0)\n            out.append(torch.diag(p) - torch.outer(p, p))\n        return torch.stack(out)\n\n    def loopC():\n        C = [[0] * 10 for _ in range(10)]\n        for a, b in zip(yt.tolist(), yp.tolist()):\n            C[a][b] += 1\n        return C\n    tJl, tJv = bench(loopJ, repeat=3, number=1), bench(lambda: ns[\"softmax_jacobian\"](Zb), repeat=3)\n    tCl, tCv = bench(loopC, repeat=3, number=1), bench(lambda: ns[\"confusion_matrix\"](yt, yp, 10), repeat=3)\n    R.table(\"Döngü vs vektörize\", [\"iş\", \"döngü\", \"vektörize\", \"kazanç\"],\n            [[\"2000 softmax Jacobian'ı (K = 10)\", fmt_time(tJl), fmt_time(tJv), f\"{tJl / tJv:,.0f}×\"],\n             [\"confusion matrix, 100 000 örnek\", fmt_time(tCl), fmt_time(tCv), f\"{tCl / tCv:,.0f}×\"]])\n"
}

In [ ]:
import importlib, labcheck, checks_m1_5_softmax as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

$K$ sınıf için model her sınıfa bir logit üretir: $z=W^\top x+b\in\mathbb R^K$. Softmax bunları bir olasılık dağılımına çevirir:

$$
p_k=\frac{e^{z_k}}{\sum_j e^{z_j}},\qquad
\ell(z,y)=-\ln p_y=\underbrace{\ln\sum_j e^{z_j}}_{\text{LSE}(z)}-z_y .
$$

Kategorik cross-entropy'nin gradyanı M1.4'teki gibi sade:

$$
\frac{\partial\ell}{\partial z}=p-e_y,\qquad
\nabla_W L=\frac1n X^\top(P-Y),\qquad \nabla_b L=\frac1n\mathbf 1^\top(P-Y),
$$

burada $P$ ve $Y$ $(n\times K)$ olasılık ve one-hot matrisleri.

Sayısal tuzak: $e^{1000}$ float64'te taşar. log-sum-exp hilesi en büyük logit'i dışarı çeker: $\text{LSE}(z)=m+\ln\sum_j e^{z_j-m}$, $m=\max z$. Artık en büyük üs $e^0=1$.

## 1 · Elle

Softmax'ı elle hesapla, sonra kaybı iki yoldan kontrol et: $-\ln p_0$ ile $\text{LSE}(z)-z_0$ aynı sayıyı vermeli.

Gradyan vektörü $p-e_y=[-0.335,\ 0.245,\ 0.090]$: bileşenleri toplamı **sıfır**. Gradient descent doğru sınıfın logit'ini yükseltir, diğerlerini kendi olasılıkları oranında düşürür.

Son grupta $z'=z+998$: softmax değişmez. Bilgisayarda $e^{1000}$'i doğrudan hesaplamaya kalkmak `inf/inf = nan` verir.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Logit'ler $z=[2,1,0]$, doğru sınıf $y=0$. Tablo: $e^2\approx7.389$, $e^1\approx2.718$**

- `Z`: $\sum_k e^{z_k}$
- `p0`: $p_0$
- `p1`: $p_1$
- `p2`: $p_2$

**Kayıp ve gradyan**

- `lse`: $\text{LSE}(z)=\ln\sum_k e^{z_k}$
- `ce`: $\ell=\text{LSE}(z)-z_y$
- `g0`: $\partial\ell/\partial z_0=p_0-1$
- `g2`: $\partial\ell/\partial z_2=p_2$

**Kararlılık: $z'=[1000,999,998]$**

- `shift`: $\max z'$ çıkarılınca logit'ler: ilk eleman
- `pp0`: $p'_0$

In [ ]:
cevaplar = dict(
    Z=None,
    p0=None,
    p1=None,
    p2=None,
    lse=None,
    ce=None,
    g0=None,
    g2=None,
    shift=None,
    pp0=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"Z":[11.107,0.003],"p0":[0.665241,0.002],"p1":[0.244728,0.002],"p2":[0.090031,0.002],"lse":[2.407606,0.003],"ce":[0.407606,0.003],"g0":[-0.334759,0.002],"g2":[0.090031,0.002],"shift":[0,0],"pp0":[0.665241,0.002]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, tek örnek. `math` serbest; `numpy`/`torch` yasak.

- `logsumexp(z)` — kararlı (max çıkararak).
- `softmax(z)` — kararlı; `logsumexp` üzerinden yazabilirsin: $p_k=e^{z_k-\text{LSE}(z)}$.
- `cross_entropy(z, y)`
- `grad_z(z, y)` — $p-e_y$ listesi.

Kontrol $z=[1000,999,998]$ ve $z=[-1000,-1001]$ ile taşma olmadığını, gradyanın sayısal türevle eşleştiğini doğrular.

In [ ]:
%%writefile stage_b.py
import math


def logsumexp(z):
    ...


def softmax(z):
    ...


def cross_entropy(z, y):
    ...


def grad_z(z, y):
    ...


print(softmax([2, 1, 0]))                  # [0.665, 0.245, 0.090]
print(cross_entropy([2, 1, 0], 0))         # 0.4076
print(softmax([1000, 999, 998]))           # aynı, taşma yok

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

CS231n'in `softmax_loss_vectorized`'ı: $n$ örnek, $K$ sınıf, **döngü yok**.

- `softmax_rows(Z)` — `(n, K)` satır satır kararlı softmax (`keepdims=True` ile max çıkar).
- `softmax_loss(W, X, y, reg=0.0)` → `(loss, dW)`; `W: (d, K)`, bias yok (X'in son sütunu 1 olabilir). Kayıp: ortalama cross-entropy $+\ \text{reg}\cdot\lVert W\rVert^2$. Doğru sınıfın log-olasılığını almak için gelişmiş indeksleme: `logP[np.arange(n), y]`. One-hot için `np.eye(K)[y]` ya da `P[np.arange(n), y] -= 1`.
- `fit_softmax(X, y, K, lr=0.5, steps=300)` → `W`; yalnız adım döngüsü.

In [ ]:
%%writefile stage_c.py
import numpy as np


def softmax_rows(Z):
    ...


def softmax_loss(W, X, y, reg=0.0):
    """→ (loss, dW). Döngü yok."""
    ...


def fit_softmax(X, y, K, lr=0.5, steps=300):
    ...


rng = np.random.default_rng(0)
centers = np.array([[0, 3], [-3, -2], [3, -2]])
y = np.repeat(np.arange(3), 50)
X = np.column_stack([centers[y] + rng.normal(size=(150, 2)), np.ones(150)])
W = fit_softmax(X, y, 3)
print("doğruluk:", (np.argmax(X @ W, 1) == y).mean())

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

`F.cross_entropy`'yi kendin yaz, sonra hazırıyla karşılaştır.

- `ce_from_logits(Z, y)` — ortalama kayıp; `torch.logsumexp(Z, dim=1)` ve doğru sınıfın logit'i için `Z.gather(1, y[:, None])` ya da `Z[torch.arange(n), y]`. `F.cross_entropy`/`log_softmax`/`nn.CrossEntropyLoss` kullanma.
- `train_softmax(X, y, K, steps=300, lr=0.5)` → `nn.Linear(d, K)`; burada `nn.CrossEntropyLoss` serbest. Logit'leri doğrudan ver.

Kontrol, `ce_from_logits`'in autograd gradyanının $(P-Y)/n$ olduğunu ve $10^4$ büyüklüğünde logit'lerde sonlu kaldığını doğrular.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F
from torch import nn


def ce_from_logits(Z, y):
    """Ortalama cross-entropy; logsumexp + gather ile."""
    ...


def train_softmax(X, y, K, steps=300, lr=0.5):
    ...


Z = torch.tensor([[2.0, 1.0, 0.0], [0.0, 0.0, 10000.0]])
y = torch.tensor([0, 2])
print(ce_from_logits(Z, y), F.cross_entropy(Z, y))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

İki yaygın "döngüyle yazılır ama yazılmamalı" işlem:

1. **Softmax Jacobian'ı.** $J=\partial p/\partial z=\text{diag}(p)-pp^\top$, $(K\times K)$. $n$ örnek için `(n, K, K)` tensörü: `torch.diag_embed(P) - P[:, :, None] * P[:, None, :]`. (Bu, `vmap(jacrev(softmax))` ile aynı şey; kapalı formu bilmek bir kopyayı ve $K$ geri geçişi kurtarır.)
2. **Confusion matrix.** $C_{ij}$: gerçek sınıfı $i$ olup $j$ tahmin edilen örnek sayısı. Döngü yerine her çifti tek bir tamsayıya kodla, $K\cdot i+j$, ve `torch.bincount(…, minlength=K*K)`.

- `softmax_jacobian(Z)` → `(n, K, K)`. **Döngü yok.**
- `confusion_matrix(y_true, y_pred, K)` → `(K, K)` tamsayı. **Döngü yok.**

In [ ]:
%%writefile stage_e.py
import torch


def softmax_jacobian(Z):
    """Z: (n, K) → (n, K, K); J[i] = diag(p) − p pᵀ. Döngü yok."""
    ...


def confusion_matrix(y_true, y_pred, K):
    """→ (K, K) tamsayı; satır gerçek, sütun tahmin. Döngü yok."""
    ...


Z = torch.tensor([[2.0, 1.0, 0.0]])
print(softmax_jacobian(Z))
print(confusion_matrix(torch.tensor([0, 0, 1, 2, 2]), torch.tensor([0, 1, 1, 2, 0]), 3))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Kaynaklar

- [CS231n — Linear classification: Softmax classifier](https://cs231n.github.io/linear-classify/#softmax) — MIT
- [CS231n Assignment 1 — softmax_loss_naive / softmax_loss_vectorized](https://cs231n.github.io/assignments2025/assignment1/) — MIT
- [Dive into Deep Learning §4.1 Softmax Regression](https://d2l.ai/chapter_linear-classification/softmax-regression.html) — CC BY-SA 4.0 / MIT-0
- [PyTorch — F.cross_entropy, torch.logsumexp](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html) — BSD-3